# 2.3 Practical Challenge: Drug Discovery for Waldenström Macroglobulinemia

Now it's your turn to act as a **computational systems pharmacologist**!

Recall from [**2.1 NF-κB Signalling in Blood Cancer**](01-nfkb-and-systems-biology.md):
- **98% of Waldenström Macroglobulinemia (WM) patients** have the ***MYD88* L265P** gain-of-function mutation.
- In our virtual B-cell, we turn on this mutation by setting `k1_MYD88SELFACTIVATION-TLR` to **`1.0`** (versus `0.0` in healthy wild-type **WT** cells).
- Your mission: **Test candidate drugs (or drug combinations!) to bring nuclear NF-κB in MYD88 L265P cells back down toward healthy WT levels.**

Click **⚡ Activate Live Editable Julia Cells** above to run and edit the simulations live!

## Step 1: Ensure the NF-κB Tri-Model Is Loaded

If you just came from Chapter 2.2 in the same session, this cell will recognise the already-compiled ODE model and finish almost instantly; if you opened Chapter 2.3 directly, it will build everything automatically for you!

In [ ]:
if isdir("/home/jovyan/work/NFkB")
    cd("/home/jovyan/work/NFkB")
elseif isdir("NFkB")
    cd("NFkB")
end
@assert isfile("combineModels.jl") "NFkB model files not found in $(pwd())."

locationOfCSV2Julia         = "csv2Julia/csv2model-multiscale.py"
moduleDefinitionFilesFolder = "moduleDefinitionFiles/"
delay                       = false

using DifferentialEquations
using Plots
using CSV
using DataFrames

if !isfile("distributedModelFiles/odeModel.jl")
    include("combineModels.jl")
    combinedModelLocation = combineModels(["BCR", "TLR", "NFkB"])
    include("generateModel.jl")
else
    parametersDF = DataFrame(CSV.File("combinedModelDefinitionFiles/parameters.csv",
                                      types=Dict(:parameter => String, :value => String)))
    println("Using already-generated Tri-Model in distributedModelFiles/odeModel.jl")
end

include("defineInputs.jl")
t = 0
include("distributedModelFiles/odeModel.jl")
include("variableNames.jl")
include("scanIncludes.jl")
include("runSimulation.jl")
include("plotAllSpecies.jl")

colorArray = palette(:seaborn_colorblind)
first_cell = 1
last_cell  = 1       # Increase (e.g. 5 or 10) to simulate heterogeneous B-cells with mean ± SD error bars & shaded ribbons!
preCV      = 0.11
maxTimeTC  = 12 * 60
maxTimeSS  = 1000 * 60
folder     = "teachingOutputs_WM"

println("Waldenström Macroglobulinemia virtual B-cell ready!")

Waldenström Macroglobulinemia virtual B-cell ready!

## Druggable Targets Reference Table

Each parameter below represents a biochemical process that a real or experimental drug can target. Pay close attention to the **direction of the multiplier**:
- To **inhibit synthesis or kinase activation**, multiply by a number **smaller than 1** (e.g. `0.1` = 90% inhibition, `0.01` = 99% inhibition).
- To **promote deactivation or boost an inhibitor (like IκBα)**, multiply by a number **greater than 1** (e.g. `5.0` or `10.0`).

| Drug Class / Mechanism | Parameter Name in Julia | Biological Effect | Typical Drug Multiplier |
| :--- | :--- | :--- | :--- |
| **IKK inhibitor** | `"k1_IKKdeactivation-linking"` | Accelerates IKK turn-off | **`10.0`** (increase deactivation) |
| **IκBα promoter / proteasome inhibitor** | `"k1_IkBaSynth-NFkB"` | Boosts IκBα negative feedback | **`10.0`** (increase inhibitor synthesis) |
| **TRAF6 / IKKK inhibitor** | `"k1_IKKKdeactivation-TLR"` | Accelerates upstream TLR/TRAF6 turn-off | **`10.0`** (increase deactivation) |
| **BTK inhibitor (BCR pathway)** | `"k1_CpBS-BCR"` | Blocks BCR → CBM signalling | **`0.1`** or **`0.01`** (inhibit activation) |
| **NIK inhibitor (Non-canonical)** | `"k1_NIKSynth-NFkB"` | Reduces NIK protein synthesis | **`0.1`** (inhibit synthesis) |
| **RelA inhibitor** | `"k1_RelASynth-NFkB"` | Reduces RelA subunit synthesis | **`0.1`** (inhibit synthesis) |
| **p50 inhibitor** | `"basal_p50Synth-NFkB"` | Reduces basal p50 synthesis | **`0.1`** (inhibit synthesis) |
| **cRel inhibitor** | `"basal_cRelSynth-NFkB"` | Reduces basal cRel synthesis | **`0.1`** (inhibit synthesis) |

## Challenge 1: Simulate MYD88 L265P & Test Your First Candidate Drug

The starter cell below sets up **healthy WT** (`k1_MYD88SELFACTIVATION-TLR = 0.0`) and **Waldenström MYD88 L265P** (`k1_MYD88SELFACTIVATION-TLR = 1.0`), plus a third condition `"myd88 l265p + drug"` where the drug multiplier is currently **`1.0` (no drug yet!)**.

**Your task:**
1. First, run the cell as-is (or immediately pick a drug from the table above!) to see how badly **MYD88 L265P** hyper-activates nuclear NF-κB compared to **WT**.
2. Then edit the 3rd condition (`"myd88 l265p + drug"`) to test a candidate drug—for example, try a **BTK inhibitor** (`"k1_CpBS-BCR"`, multiplier `0.1`), an **IκBα promoter** (`"k1_IkBaSynth-NFkB"`, multiplier `10.0`), or an **IKK inhibitor** (`"k1_IKKdeactivation-linking"`, multiplier `10.0`).
3. Which drugs actually work against **MYD88 L265P**, and why might a **pure BCR-pathway BTK inhibitor** behave differently from a **downstream IKK / IκBα** target in this model?

In [ ]:
# Challenge 1 starter code — edit the 3rd condition to test your chosen drug!
conditions     = []
paramsToChange = []
modifyAmount   = []
BCRSSArray     = []
BCRTCArray     = []
TLRSSArray     = []
TLRTCArray     = []
NIKSSArray     = []
NIKTCArray     = []

# 1. Healthy WT B-cell
push!(conditions,     "WT")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR"])
push!(modifyAmount,   [0.0])
push!(BCRSSArray,     bcrSignalSS)
push!(BCRTCArray,     bcrSignalSS)
push!(TLRSSArray,     tlrSignalSS)
push!(TLRTCArray,     tlrSignalTC)
push!(NIKSSArray,     nikSignalSS)
push!(NIKTCArray,     nikSignalSS)

# 2. Waldenström Macroglobulinemia (MYD88 L265P mutation ON)
push!(conditions,     "myd88 l265p")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR"])
push!(modifyAmount,   [1.0])
push!(BCRSSArray,     bcrSignalSS)
push!(BCRTCArray,     bcrSignalSS)
push!(TLRSSArray,     tlrSignalSS)
push!(TLRTCArray,     tlrSignalTC)
push!(NIKSSArray,     nikSignalSS)
push!(NIKTCArray,     nikSignalSS)

# 3. MYD88 L265P + YOUR DRUG (currently multiplier = 1.0 -> no drug yet!)
push!(conditions,     "myd88 l265p + drug")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR", "k1_IkBaSynth-NFkB"])  # <-- Pick any drug target!
push!(modifyAmount,   [1.0, 1.0])                                           # <-- Change 1.0 to your drug dose (e.g. 10.0 or 0.1)!
push!(BCRSSArray,     bcrSignalSS)
push!(BCRTCArray,     bcrSignalSS)
push!(TLRSSArray,     tlrSignalSS)
push!(TLRTCArray,     tlrSignalTC)
push!(NIKSSArray,     nikSignalSS)
push!(NIKTCArray,     nikSignalSS)

display(DataFrame(
    Condition  = conditions,
    Parameters = [join(x, ", ") for x in paramsToChange],
    Multiplier = [join(string.(x), ", ") for x in modifyAmount],
))

runSimulation(
    first_cell, last_cell, conditions, folder,
    BCRSSArray, BCRTCArray, TLRSSArray, TLRTCArray, NIKSSArray, NIKTCArray
)

speciesToPlot  = ["RelAnp50n", "cRelnp50n", "RelBnp52n"]
gr()
plotAllSpecies(speciesToPlot, conditions, colorArray, first_cell, last_cell, folder, 12, 0, 0.5, 60)

````{admonition} 💡 Hint — Challenge 1
:class: tip, dropdown

Look at the **Tri-Model diagram** from Chapter 2.2:
- **MYD88 L265P** sits on the **TLR branch** (`MYD88 → TRAF6 → TAK1 → IKK → IκBα/NF-κB`).
- Because `bcrSignalSS = 0` here, blocking the **BCR branch** (`"k1_CpBS-BCR"`) does not intercept the constitutive MYD88 → TRAF6 → TAK1 → IKK signal!
- Try boosting **IκBα negative feedback** (`"k1_IkBaSynth-NFkB"` with multiplier `10.0`) or inhibiting **IKK** (`"k1_IKKdeactivation-linking"` with multiplier `10.0`)—keep `1.0` as the first entry in `modifyAmount` so the MYD88 L265P mutation stays ON!
````

````{admonition} ✅ Reveal Solution — Challenge 1
:class: note, dropdown

Boosting IκBα synthesis (`"k1_IkBaSynth-NFkB"` × `10.0`) pulls canonical `RelAnp50n` and `cRelnp50n` strongly back down toward healthy WT levels, even with the MYD88 L265P mutation active!

```julia
conditions     = []
paramsToChange = []
modifyAmount   = []
BCRSSArray     = []; BCRTCArray = []
TLRSSArray     = []; TLRTCArray = []
NIKSSArray     = []; NIKTCArray = []

# 1. Healthy WT
push!(conditions,     "WT")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR"])
push!(modifyAmount,   [0.0])
push!(BCRSSArray, bcrSignalSS); push!(BCRTCArray, bcrSignalSS)
push!(TLRSSArray, tlrSignalSS); push!(TLRTCArray, tlrSignalTC)
push!(NIKSSArray, nikSignalSS); push!(NIKTCArray, nikSignalSS)

# 2. MYD88 L265P (untreated)
push!(conditions,     "myd88 l265p")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR"])
push!(modifyAmount,   [1.0])
push!(BCRSSArray, bcrSignalSS); push!(BCRTCArray, bcrSignalSS)
push!(TLRSSArray, tlrSignalSS); push!(TLRTCArray, tlrSignalTC)
push!(NIKSSArray, nikSignalSS); push!(NIKTCArray, nikSignalSS)

# 3. MYD88 L265P + IκBα promoter (10x)
push!(conditions,     "myd88 l265p + IkBa drug")
push!(paramsToChange, ["k1_MYD88SELFACTIVATION-TLR", "k1_IkBaSynth-NFkB"])
push!(modifyAmount,   [1.0, 10.0])
push!(BCRSSArray, bcrSignalSS); push!(BCRTCArray, bcrSignalSS)
push!(TLRSSArray, tlrSignalSS); push!(TLRTCArray, tlrSignalTC)
push!(NIKSSArray, nikSignalSS); push!(NIKTCArray, nikSignalSS)

runSimulation(first_cell, last_cell, conditions, folder,
              BCRSSArray, BCRTCArray, TLRSSArray, TLRTCArray, NIKSSArray, NIKTCArray)

gr()
plotAllSpecies(["RelAnp50n", "cRelnp50n", "RelBnp52n"], conditions, colorArray,
               first_cell, last_cell, folder, 12, 0, 0.5, 60)
```
````

## Challenge 2: Drug Screen & Combination Therapy

In real oncology, single drugs often leave one arm of signalling active (for example, suppressing canonical **RelA:p50** while leaving non-canonical **RelB:p52** elevated, or requiring high toxic doses).

**Your task:** Add a **4th condition** to the starter cell below to compare **two different drugs side by side**—or test a **combination therapy** (two drug parameters inside the same `paramsToChange` array, e.g. combining an **IKK inhibitor** with a **NIK inhibitor** or **cRel inhibitor**)!

````{admonition} 📋 Quick Reference: Druggable Targets Table
:class: note, dropdown

- **Inhibit synthesis / activation:** multiplier `< 1.0` (e.g. `0.1`)
- **Boost deactivation / IκBα inhibitor:** multiplier `> 1.0` (e.g. `5.0` or `10.0`)

| Drug Class / Mechanism | Parameter Name in Julia | Biological Effect | Typical Drug Multiplier |
| :--- | :--- | :--- | :--- |
| **IKK inhibitor** | `"k1_IKKdeactivation-linking"` | Accelerates IKK turn-off | **`10.0`** (increase deactivation) |
| **IκBα promoter / proteasome inhibitor** | `"k1_IkBaSynth-NFkB"` | Boosts IκBα negative feedback | **`10.0`** (increase inhibitor synthesis) |
| **TRAF6 / IKKK inhibitor** | `"k1_IKKKdeactivation-TLR"` | Accelerates upstream TLR/TRAF6 turn-off | **`10.0`** (increase deactivation) |
| **BTK inhibitor (BCR pathway)** | `"k1_CpBS-BCR"` | Blocks BCR → CBM signalling | **`0.1`** or **`0.01`** (inhibit activation) |
| **NIK inhibitor (Non-canonical)** | `"k1_NIKSynth-NFkB"` | Reduces NIK protein synthesis | **`0.1`** (inhibit synthesis) |
| **RelA inhibitor** | `"k1_RelASynth-NFkB"` | Reduces RelA subunit synthesis | **`0.1`** (inhibit synthesis) |
| **p50 inhibitor** | `"basal_p50Synth-NFkB"` | Reduces basal p50 synthesis | **`0.1`** (inhibit synthesis) |
| **cRel inhibitor** | `"basal_cRelSynth-NFkB"` | Reduces basal cRel synthesis | **`0.1`** (inhibit synthesis) |
````

In [ ]:
# Challenge 2 starter code — add a 4th condition (a competing drug or a 2-drug combination!)
conditions     = []
paramsToChange = []
modifyAmount   = []
BCRSSArray     = []; BCRTCArray = []
TLRSSArray     = []; TLRTCArray = []
NIKSSArray     = []; NIKTCArray = []

function add_wm_condition!(name, params, mults)
    push!(conditions,     name)
    push!(paramsToChange, params)
    push!(modifyAmount,   mults)
    push!(BCRSSArray, bcrSignalSS); push!(BCRTCArray, bcrSignalSS)
    push!(TLRSSArray, tlrSignalSS); push!(TLRTCArray, tlrSignalTC)
    push!(NIKSSArray, nikSignalSS); push!(NIKTCArray, nikSignalSS)
end

add_wm_condition!("WT",                ["k1_MYD88SELFACTIVATION-TLR"],                               [0.0])
add_wm_condition!("MYD88 L265P",       ["k1_MYD88SELFACTIVATION-TLR"],                               [1.0])
add_wm_condition!("L265P + Drug A",    ["k1_MYD88SELFACTIVATION-TLR", "k1_IKKdeactivation-linking"], [1.0, 5.0])
# <-- Add a 4th line below using add_wm_condition!(...) for Drug B or a combination of Drug A + Drug B!

display(DataFrame(
    Condition  = conditions,
    Parameters = [join(x, ", ") for x in paramsToChange],
    Multiplier = [join(string.(x), ", ") for x in modifyAmount],
))

runSimulation(first_cell, last_cell, conditions, folder,
              BCRSSArray, BCRTCArray, TLRSSArray, TLRTCArray, NIKSSArray, NIKTCArray)

gr()
plotAllSpecies(["RelAnp50n", "cRelnp50n", "RelBnp52n"], conditions, colorArray,
               first_cell, last_cell, folder, 12, 0, 0.5, 60)

````{admonition} 💡 Hint — Challenge 2
:class: tip, dropdown

You can pass **two** drug targets alongside `"k1_MYD88SELFACTIVATION-TLR"` in the same `add_wm_condition!` call—for example:
```julia
add_wm_condition!(
    "L265P + IKK + NIK combo",
    ["k1_MYD88SELFACTIVATION-TLR", "k1_IKKdeactivation-linking", "k1_NIKSynth-NFkB"],
    [1.0, 5.0, 0.2],
)
```
Watch what happens to **both** canonical (`RelAnp50n`, `cRelnp50n`) **and** non-canonical (`RelBnp52n`) nuclear NF-κB!
````

````{admonition} ✅ Reveal Solution — Challenge 2
:class: note, dropdown

Comparing moderate single-agent IKK inhibition (`5×`) against a dual **IKK + NIK** combination (`k1_IKKdeactivation-linking` × `5.0` plus `k1_NIKSynth-NFkB` × `0.2`) shows how targeting both pathways simultaneously lowers canonical **RelA/cRel** *and* suppresses non-canonical **RelB:p52**!

```julia
conditions     = []
paramsToChange = []
modifyAmount   = []
BCRSSArray     = []; BCRTCArray = []
TLRSSArray     = []; TLRTCArray = []
NIKSSArray     = []; NIKTCArray = []

function add_wm_condition!(name, params, mults)
    push!(conditions,     name)
    push!(paramsToChange, params)
    push!(modifyAmount,   mults)
    push!(BCRSSArray, bcrSignalSS); push!(BCRTCArray, bcrSignalSS)
    push!(TLRSSArray, tlrSignalSS); push!(TLRTCArray, tlrSignalTC)
    push!(NIKSSArray, nikSignalSS); push!(NIKTCArray, nikSignalSS)
end

add_wm_condition!("WT",                       ["k1_MYD88SELFACTIVATION-TLR"],                                                    [0.0])
add_wm_condition!("MYD88 L265P",              ["k1_MYD88SELFACTIVATION-TLR"],                                                    [1.0])
add_wm_condition!("L265P + IKK (5x)",         ["k1_MYD88SELFACTIVATION-TLR", "k1_IKKdeactivation-linking"],                      [1.0, 5.0])
add_wm_condition!("L265P + IKK + NIK combo",  ["k1_MYD88SELFACTIVATION-TLR", "k1_IKKdeactivation-linking", "k1_NIKSynth-NFkB"],  [1.0, 5.0, 0.2])

runSimulation(first_cell, last_cell, conditions, folder,
              BCRSSArray, BCRTCArray, TLRSSArray, TLRTCArray, NIKSSArray, NIKTCArray)

gr()
plotAllSpecies(["RelAnp50n", "cRelnp50n", "RelBnp52n"], conditions, colorArray,
               first_cell, last_cell, folder, 12, 0, 0.5, 60)
```
````

## Assessment & Discussion Ideas

Looking for a **6-minute SSC presentation** topic from this session? Consider exploring:

1. **Drug Resistance in Waldenström Macroglobulinemia:** Compare why a BCR-focused BTK inhibitor (`k1_CpBS-BCR`) behaves differently when basal BCR signalling is LOW (`bcrSignalSS`) versus HIGH (`bcrSignalSSHigh`) in a MYD88 L265P cell!
2. **Toxicity vs Efficacy Trade-Off:** What happens if you apply your best MYD88 L265P drug to a **healthy WT cell**? Does it wipe out normal immune TLR signalling?
3. **Dose-Response Curve:** Run your favourite drug at `1×`, `2×`, `5×`, and `10×` (or `0.5×`, `0.2×`, `0.05×`) to find the *minimum effective dose* that restores WT-like NF-κB levels.
4. **Exploring Other Species:** Try changing `speciesToPlot` to inspect upstream proteins like `["IKK_on", "TAK1p", "A20", "IkBa"]`—you can inspect all 140 species by printing `syms`!